In [1]:
# Email spam classification using Multinomial Naive Bayes
import pandas as pd

In [4]:
df = pd.read_csv("data/email.csv")

In [5]:
# DATA UNDERSTANDING

print(df.info())
print(df.describe())
print(df["Category"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5573 entries, 0 to 5572
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5573 non-null   object
 1   Message   5573 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None
       Category                 Message
count      5573                    5573
unique        3                    5158
top         ham  Sorry, I'll call later
freq       4825                      30
Category
ham               4825
spam               747
{"mode":"full"       1
Name: count, dtype: int64


In [6]:
# DATA CLEANING

# remove last row that is not related to dataset
df.drop(5572, inplace=True)

In [7]:
print(df.shape)

# strip messages
df["Message"] = df["Message"].str.strip()

# remove messages with empty text
df.dropna(inplace=True)
df = df[df['Message'] != ""]

# turn them all to lowercase
df["Message"] = df["Message"].str.lower()

# turn long whitespace to single whitespace
df['Message'] = df['Message'].str.replace(r'\s+', ' ', regex=True).str.strip()

# reset index order
df.reset_index(drop=True, inplace=True)

# remove duplicates
df.drop_duplicates(inplace=True, ignore_index=True)

print(df.shape)

(5572, 2)
(5156, 2)


In [8]:
# Map categories to 0 = ham and 1 = spam
categories = {"spam": 1, "ham": 0}
df["Category"] = df["Category"].map(categories)

In [9]:
# MODELING

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# split data
X_train, X_test, y_train, y_test = train_test_split(df["Message"], df["Category"], test_size=0.2, random_state=42, stratify=df["Category"])

# convert text to numbers
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

# train model
model = MultinomialNB(alpha=1)
model.fit(X_train, y_train)

MultinomialNB(alpha=1)

In [11]:
# EVALUATION
y_pred = model.predict(X_test)
print(y_pred)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:", classification_report(y_test, y_pred))

[0 0 0 ... 1 0 0]
Accuracy: 0.9825581395348837
Classification Report:               precision    recall  f1-score   support

           0       0.98      1.00      0.99       904
           1       0.97      0.88      0.93       128

    accuracy                           0.98      1032
   macro avg       0.98      0.94      0.96      1032
weighted avg       0.98      0.98      0.98      1032



In [13]:
from sklearn.pipeline import Pipeline
import joblib

pipeline = Pipeline([
    ('vectorizer', vectorizer),
    ('model', model)
])

bundle = {
    "pipeline": pipeline,
    "metadata": {
    "model_name": "spam_classifier",
    "model_version": "1.0.0",
    "framework": "scikit-learn",
    "features": "Message",
    "target": "label",
    "created_at": "2026-09-16"
    }
}

joblib.dump(bundle, 'model.joblib')

['model.joblib']